# Overton policy document classification

Applies the fine-tuned SciBERT classifier to Overton policy documents, producing a 21-field probability distribution per document.

**Input:** `overton_data/overton_merged.jsonl` (200,030 documents)  
**Model:** fine-tuned SciBERT classifier (`scibert_classifier/`)  
**Output:** `policy_classified.jsonl` — per-document field probabilities  
**Text used:** `title` + `llm_document_description`

## Install dependencies

In [ ]:
%pip install transformers torch pandas tqdm --quiet

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

In [ ]:
import json
import os
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Paths

Adjust to match your environment.

In [ ]:
import shutil

DRIVE_MODEL_DIR = "/content/drive/MyDrive/Colab Notebooks/scibert_classifier"
LOCAL_MODEL_DIR = "/content/scibert_classifier"

if not os.path.exists(LOCAL_MODEL_DIR):
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)

In [ ]:
MODEL_DIR = LOCAL_MODEL_DIR
INPUT_PATH = "/content/drive/MyDrive/Colab Notebooks/overton_merged.jsonl"
OUTPUT_PATH = "/content/drive/MyDrive/Colab Notebooks/policy_classified.jsonl"
CHECKPOINT = "/content/drive/MyDrive/Colab Notebooks/classify_checkpoint.txt"
BATCH_SIZE = 16

## Load model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, local_files_only=True).to(DEVICE)
model.eval()

id2label = model.config.id2label
labels = [id2label[i] for i in range(len(id2label))]
print(f"Loaded model with {len(labels)} labels")

## Classification functions

In [ ]:
def make_text(doc):
    """Concatenate title and llm_document_description; fall back to title only."""
    title = doc.get("title") or ""
    desc = doc.get("llm_document_description") or ""
    if desc:
        return f"{title}. {desc}".strip()
    return title.strip()


def classify_batch(texts):
    """Returns a (batch_size, 21) array of softmax probabilities."""
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt",
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model(**encodings)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    return probs

## Classify all documents

Checkpointed for resumability; can take several hours on CPU.

In [ ]:
start_idx = 0
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        start_idx = int(f.read().strip()) + 1
    print(f"Resuming from document {start_idx}")
else:
    print("Starting from the beginning")

In [ ]:
with open(INPUT_PATH) as f:
    all_docs = [json.loads(line) for line in f]

docs_to_process = all_docs[start_idx:]
print(f"to process: {len(docs_to_process):,} / total {len(all_docs):,}")

In [ ]:
def write_result(out_f, doc, prob):
    result = {
        "policy_document_id": doc.get("policy_document_id"),
        "asean_code": doc.get("asean_code"),
        "source_type": doc.get("source_type"),
        "title": doc.get("title"),
        "published_on": doc.get("published_on"),
        "pred_label": labels[int(np.argmax(prob))],
        "pred_probs": {l: float(p) for l, p in zip(labels, prob)},
    }
    out_f.write(json.dumps(result, ensure_ascii=False) + "\n")


skipped = 0
with open(OUTPUT_PATH, "a", encoding="utf-8") as out_f:
    batch_docs, batch_texts = [], []
    for i, doc in enumerate(tqdm(docs_to_process, desc="classifying")):
        text = make_text(doc)
        if not text:
            skipped += 1
            continue
        batch_docs.append(doc)
        batch_texts.append(text)

        if len(batch_texts) == BATCH_SIZE:
            probs = classify_batch(batch_texts)
            for doc_, prob in zip(batch_docs, probs):
                write_result(out_f, doc_, prob)
            out_f.flush()

            global_idx = start_idx + i
            with open(CHECKPOINT, "w") as cp:
                cp.write(str(global_idx))

            batch_docs, batch_texts = [], []

    if batch_texts:
        probs = classify_batch(batch_texts)
        for doc_, prob in zip(batch_docs, probs):
            write_result(out_f, doc_, prob)

print(f"done. skipped: {skipped:,}")

## Spot-check a sample result

In [ ]:
with open(OUTPUT_PATH) as f:
    sample = json.loads(f.readline())

print("title:", sample["title"][:60])
print("asean_code:", sample["asean_code"])
print("pred_label:", sample["pred_label"])

top5 = sorted(sample["pred_probs"].items(), key=lambda x: -x[1])[:5]
for label, prob in top5:
    print(f"  {label:45s} {prob:.3f}")

## Sanity check: top fields by country

Quick read of the classified output to confirm the distributions look reasonable before moving to `4_alignment_analysis.ipynb`, which computes the full 21-field distributions and alignment scores.

In [ ]:
from collections import defaultdict, Counter

country_field_counts = defaultdict(Counter)
with open(OUTPUT_PATH) as f:
    for line in f:
        doc = json.loads(line)
        code = doc.get("asean_code") or "INTL"
        label = doc.get("pred_label")
        if label:
            country_field_counts[code][label] += 1

for code in sorted(country_field_counts.keys(), key=lambda x: x or ""):
    total = sum(country_field_counts[code].values())
    print(f"\n[{code}] total {total:,}")
    for field, cnt in country_field_counts[code].most_common(5):
        print(f"  {field:45s} {cnt:>5,} ({cnt / total * 100:.1f}%)")